# 01 · Data merging & cleaning
Builds the canonical NFHS-5 district table and the pooled HMIS maternal-health table, reconciles district names, and engineers features (blueprint §2). Re-extraction of the HMIS archive takes several minutes, so it is skipped when the interim CSV already exists.

In [1]:
import sys, pathlib, json, warnings
warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src.configuration import load_config, configured_path, feature_columns, zones
config = load_config()
P = lambda name: configured_path(config, name)
pd.set_option("display.max_columns", 40)

In [2]:
from src.data.nfhs_extract import build_nfhs_canonical
from src.data.hmis_extract import pool_hmis_annual_files
REBUILD = False
if REBUILD or not P("nfhs_raw").exists():
    build_nfhs_canonical(P("nfhs_source"), P("nfhs_raw"))
hmis_files = [P(f"hmis_{y}") for y in config["real_data"]["hmis_years"]]
if REBUILD or not P("hmis_raw").exists():
    pool_hmis_annual_files(hmis_files, P("hmis_raw"), progress=True)
nfhs, hmis = pd.read_csv(P("nfhs_raw")), pd.read_csv(P("hmis_raw"))
print("NFHS districts:", len(nfhs), "| HMIS districts:", len(hmis), "| HMIS archives present:", [f.name for f in hmis_files if f.exists()])

NFHS districts: 707 | HMIS districts: 704 | HMIS archives present: ['hmis_2019_20_official.zip']


## District reconciliation
Within-state matching: aliases → exact normalised name → `rapidfuzz` token-sort (≥ 90 auto, 75–89 manual review). HMIS districts carved out after NFHS-5 are summed into their NFHS parent.

In [3]:
from src.data.merge_nfhs_hmis import merge_sources
m = config["matching"]
master = merge_sources(P("nfhs_raw"), P("hmis_raw"), ROOT / m["aliases_path"], P("match_audit"),
                       m["automatic_match_threshold"], m["review_match_threshold"], config["real_data"]["minimum_matched_share"])
audit = pd.read_csv(P("match_audit"))
print(f"Matched share: {master['hmis_matched'].mean():.1%}")
audit["method"].value_counts()

Matched share: 98.9%


method
exact_normalised_name    671
fuzzy_auto                27
unmatched                  8
hmis_unused                2
Name: count, dtype: int64

In [4]:
audit.loc[audit["method"].isin(["unmatched", "manual_review", "hmis_unused", "fuzzy_auto"])]

,nfhs_state,nfhs_district,hmis_state,hmis_district,method,similarity_score,best_unaccepted_candidate
12,Andhra Pradesh,Visakhapatnam,Andhra Pradesh,Vishakapatnam,fuzzy_auto,92.307692,NaN
38,Assam,Biswanath,NaN,NaN,unmatched,NaN,NaN
41,Assam,Charaideo,NaN,NaN,unmatched,NaN,NaN
51,Assam,Hojai,NaN,NaN,unmatched,NaN,NaN
59,Assam,Majuli,NaN,NaN,unmatched,NaN,NaN
65,Assam,South Salmara Mancachar,NaN,NaN,unmatched,NaN,NaN
68,Assam,West Karbi Anglong,NaN,NaN,unmatched,NaN,NaN
112,Chhattisgarh,Bemetara,Chhattisgarh,Bemetra,fuzzy_auto,93.333333,NaN
122,Chhattisgarh,Kodagaon,Chhattisgarh,Kondagaon,fuzzy_auto,94.117647,NaN
143,Gujarat,Aravali,Gujarat,Aravalli,fuzzy_auto,93.333333,NaN


## Feature engineering & missingness

In [5]:
from src.data.preprocess_impute import engineer_features
master = engineer_features(master, feature_columns(), config["preprocessing"])
missing = master[feature_columns()].isna().mean().sort_values(ascending=False)
missing.head(10).to_frame("missing share")

,missing share
private_c_section_pct,0.213881
pregnant_anaemia_pct,0.189802
hmis_severe_anaemia_pct,0.011331
hmis_jssk_transport_pct,0.011331
hmis_ifa_180_pct,0.011331
hmis_early_discharge_pct,0.011331
hmis_first_trimester_reg_pct,0.011331
hmis_blood_units_per_1000_deliveries,0.011331
hmis_c_section_rate_pct,0.011331
oop_expenditure_log,0.001416


In [6]:
master[feature_columns()].describe().T[["count", "mean", "50%", "min", "max"]].round(2)

,count,mean,50%,min,max
public_c_section_pct,706.0,17.28,12.95,0.70,73.00
public_c_section_deficit,706.0,1.76,0.00,0.00,9.30
private_c_section_pct,555.0,49.53,46.20,6.30,94.20
skilled_birth_attendance_pct,706.0,89.59,92.30,30.90,100.00
hmis_c_section_rate_pct,698.0,20.58,16.38,0.00,74.88
hmis_blood_units_per_1000_deliveries,698.0,155.38,53.65,0.00,3011.28
hmis_jssk_transport_pct,698.0,27.94,22.09,0.00,100.00
hmis_early_discharge_pct,698.0,35.15,27.39,0.00,100.00
pregnant_anaemia_pct,572.0,50.46,52.10,2.00,87.60
all_women_anaemia_pct,706.0,55.95,57.20,14.90,93.50
